# 第五部分：DeepAgent Skills 与渐进式披露 (Progressive Disclosure)
### *Alex 的终极方案：拥有“翻书”能力的智能体*

Alex 终于意识到，把所有规则塞进脑子（Mega-Prompt）是行不通的。他采用了 DeepAgent 的 **Skills** 架构。

**核心原理：**
1. **元数据注入 (Metadata Injection)**：Agent 初始化时只加载技能的名字和描述。这就像给 Agent 一张**“技能目录”**。
2. **按需读取 (On-demand Reading)**：只有当 Agent 认为某个技能对当前任务有用时，它才会调用内置工具去读取 `SKILL.md` 的**“详细手册”**。

在这个笔记本中，我们将：
1. **实现流式追踪器**：实时截获并打印 Agent 接收到的 System Prompt，看它是如何动态变化的。
2. **流式任务执行 (Streaming)**：由于复杂任务运行时间较长，我们将开启流式模式，实时观察 Agent 的思考轨迹。
3. **验证性能**：看 Agent 如何在不浪费 Token 的情况下，精准完成财务与市场的跨部门分析。

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from deepagents.middleware.skills import SKILLS_SYSTEM_PROMPT

load_dotenv(override=True)

# 1. 准备技能目录
os.makedirs("skills/finance_pro", exist_ok=True)
os.makedirs("skills/market_wizard", exist_ok=True)

print("=== DeepAgent Skills Middleware 自动注入的提示词 ===\n")
print(SKILLS_SYSTEM_PROMPT)
print("=" * 60)


## 2. 初始化流式 Agent
我们将开启流式模式，这样当 Agent 在进行长链条推理时，我们能实时看到输出。


In [ ]:
model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
    streaming=True
)

root_dir = Path.cwd()
backend = FilesystemBackend(root_dir=str(root_dir))

repl = PythonREPL()

@tool
def python_analyst(code: str):
    """Execute Python analysis. Has access to enterprise_data/ files."""
    return repl.run(code)

@tool
def generate_report(csv_path: str):
    """
    Generate a professional GlobalCorp visual report (bar + pie charts) for any CSV.
    The CSV should have columns for: region, quantity, unit price, product category, and discount.
    Handles both standard (Region, Quantity, Discount) and cleaned naming conventions.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False

    df = pd.read_csv(csv_path)
    col_map = {c: c.lower().strip() for c in df.columns}
    reverse_map = {v: k for k, v in col_map.items()}

    region_col = reverse_map.get("region")
    qty_col = reverse_map.get("quantity") or reverse_map.get("qty")
    price_col = reverse_map.get("unit_price") or reverse_map.get("price_per_unit") or reverse_map.get("unit price")
    cat_col = reverse_map.get("product_category") or reverse_map.get("prod_cat") or reverse_map.get("product category")
    discount_col = reverse_map.get("discount") or reverse_map.get("rebate")

    if not all([region_col, qty_col, price_col, cat_col]):
        return f"错误：CSV 缺少必要列。找到的列: {list(df.columns)}"

    df[price_col] = df[price_col].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)

    if discount_col:
        df["Total_Revenue"] = (df[qty_col].astype(float) * df[price_col]) - df[discount_col].astype(float)
    else:
        df["Total_Revenue"] = df[qty_col].astype(float) * df[price_col]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    region_data = df.groupby(region_col)["Total_Revenue"].sum().sort_values(ascending=False)
    sns.barplot(x=region_data.index, y=region_data.values, hue=region_data.index, ax=ax1, palette="viridis", legend=False)
    ax1.set_title("按地区汇总的净收入", fontsize=14, fontweight="bold")
    ax1.set_ylabel("收入 (USD)")

    cat_data = df.groupby(cat_col)["Total_Revenue"].sum()
    ax2.pie(cat_data, labels=cat_data.index, autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("pastel"), wedgeprops={"edgecolor": "white", "linewidth": 2})
    centre_circle = plt.Circle((0, 0), 0.70, fc="white")
    ax2.add_artist(centre_circle)
    ax2.set_title("产品类别贡献率", fontsize=14, fontweight="bold")

    plt.suptitle(f"GlobalCorp 业务分析报告: {csv_path}", fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    return "报告生成成功。"

# 注意：不传 system_prompt，让 middleware 自动注入技能提示
agent = create_deep_agent(
    model=model,
    tools=[python_analyst, generate_report],
    backend=backend,
    skills=[str(root_dir / "skills")],
)


## 4. 执行并观察：渐进式披露的实况转播
请注意观察：
1. **第一步**：你会看到 LLM 的输入中包含了技能的 `name` 和 `description`，但没有具体规则。
2. **第二步**：你会看到 Agent 发现需要财务规则，调用了 `read_skill`。
3. **第三步**：再次发送给 LLM 的输入中，`finance_pro` 的全文被加载了进来。

In [ ]:
from utils import stream_agent
import asyncio

query = "请分析 modern_marketing.csv 并给我一份完整的分析报告"

await stream_agent(
    agent,
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 50},
)


## 5. 结论：为什么 Alex 终于成功了？

观察追踪日志，你会发现 Skills 模式的三个核心优势：

1. **Token 效率**：初始请求非常短（只有目录）。比起 Notebook 4 的全量注入，我们节省了大量的“背景 Token”。
2. **零干扰**：如果用户问的是天气，Agent 根本不会去读财务手册，从而避免了不相关规则带来的幻觉。
3. **解耦与动态性**：如果你现在去修改 `skills/finance_pro/SKILL.md`，Agent 在下一次调用 `read_skill` 时会立刻获得新知识，而无需重启或重新训练模型。

### Alex 的最终寄语：
“不要试图让 Agent 成为一个全知全能的神。要让它成为一个善于学习、懂得按需获取知识的**专业工作者**。这就是 DeepAgent Skills 的真谛。”